# Calibrate H5 File Workflow

Run HDF5 calibration and inspect the fitted per-channel timing/lifetime outputs.


In [ ]:
from pathlib import Path
import sys

import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

repo_root = Path.cwd().resolve()
while not (repo_root / "src").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent

src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from brighteyes_mcs_file import calibrate_h5_file, show_h5_structure_html
import brighteyes_mcs_file.graph as graph

EXPECTED_H5_DATA_FORMAT_VERSION = "0.0.6"


def assert_h5_schema_version(filename, expected=EXPECTED_H5_DATA_FORMAT_VERSION):
    with h5py.File(filename, "r") as hf:
        actual = hf.attrs["data_format_version"]
    actual = actual.decode() if isinstance(actual, bytes) else str(actual)
    if actual != expected:
        raise ValueError(f"Expected H5 schema version {expected}, found {actual} in {filename}")
    return actual


## Calibration Parameters


In [ ]:
data_key = "data"
data_keys = (data_key,) if isinstance(data_key, str) else tuple(data_key)
inspect_data_key = data_keys[0]

data_folder = Path(
    "/mnt/DATA/Mixed Data/26-03-17_Convallaria_and_FLIMLABS_calibrated_Yellow_Slide"
)
measurement_file = data_folder / "03_Convallaria_DFD_40MHz-17-03-2026-17-07-20.h5"
reference_file = data_folder / "FLIMLABS_Yellow_slide_2_5ns-17-03-2026-16-18-22.h5"

tau_reference_ns = 2.5
reference_type = "ref"
fit_mode = "model_shift"
fit_type = "likelihood"
laser_period_ns = None

channel_skew = {
    "type": "phase_cross_correlation",
    "source": "ref",
    "reference_channel": 12,
    "upsampling": 10,
    "apodize": False,
}

overwrite_existing = True


## Run Calibration


In [ ]:
output_path = calibrate_h5_file(
    measurement_file,
    reference_file,
    data_key=data_key,
    reference_type=reference_type,
    tau_ref=tau_reference_ns,
    fit_mode=fit_mode,
    fit_type=fit_type,
    channel_skew_type=channel_skew["type"],
    channel_skew_source=channel_skew["source"],
    channel_skew_fit_reference_channel=channel_skew["reference_channel"],
    channel_skew_fit_upsampling=channel_skew["upsampling"],
    channel_skew_fit_apodize=channel_skew["apodize"],
    period_ns=laser_period_ns,
    overwrite=overwrite_existing,
    create_output=True,
)

output_path = Path(output_path)
h5_schema_version = assert_h5_schema_version(output_path)
output_path


## Load Calibration Tables


In [ ]:
PRODUCT_NAME_BY_DATA_KEY = {
    "data": "spad",
    "spad": "spad",
    "raw/spad": "spad",
    "data_channels_extra": "aux",
    "aux": "aux",
    "raw/aux": "aux",
}

SUMMARY_DATASETS = {
    "channel": "channels/index",
    "reference_mask": "channels/reference_mask",
    "amplitude": "fit/amplitude",
    "amplitude_err": "fit/amplitude_err",
    "tau_ns": "fit/tau_ns",
    "tau_err_ns": "fit/tau_err_ns",
    "tau_reference_ns": "fit/tau_reference_ns",
    "fitted_delay_bins": "fit/fitted_delay_bins",
    "fitted_delay_ns": "fit/fitted_delay_ns",
    "delay_correction_bins": "timing/delay_correction_bins",
    "delay_correction_err_bins": "timing/delay_correction_err_bins",
    "fitted_delay_err_bins": "fit/fitted_delay_err_bins",
    "delay_correction_ns": "timing/delay_correction_ns",
    "delay_correction_err_ns": "timing/delay_correction_err_ns",
    "fitted_delay_err_ns": "fit/fitted_delay_err_ns",
    "channel_skew": "timing/channel_skew_bins",
    "channel_skew_err": "timing/channel_skew_err_bins",
    "residual_error": "fit/residual_error",
}


def calibration_product_name(source_key):
    normalized_key = str(source_key).strip("/")
    return PRODUCT_NAME_BY_DATA_KEY[normalized_key]


def calibration_result_group(handle, source_key=inspect_data_key, calibration_key="calibration"):
    product_name = calibration_product_name(source_key)
    results_path = f"{calibration_key}/results"
    result_path = f"{results_path}/{product_name}"
    return handle[result_path]


def read_calibration_summary(result_group):
    return pd.DataFrame(
        {
            column_name: result_group[dataset_path][()]
            for column_name, dataset_path in SUMMARY_DATASETS.items()
        }
    )


with h5py.File(output_path, "r") as h5_file:
    result_group = calibration_result_group(h5_file)
    summary_df = read_calibration_summary(result_group)

summary_df


## Channel Fit Summary


In [ ]:
graph.plot_calibration_lifetime_summary(summary_df)


In [ ]:
summary_tables = []
plot_labels = []

with h5py.File(output_path, "r") as h5_file:
    for source_key in data_keys:
        result_group = calibration_result_group(h5_file, source_key=source_key)
        summary_tables.append(read_calibration_summary(result_group))

        skew_source = result_group.attrs["channel_skew_source"]
        reference_key = result_group.attrs["channel_skew_fit_reference_data_key"]
        reference_channel = result_group.attrs["channel_skew_fit_reference_channel_resolved"]
        plot_labels.append(f"{source_key} | {skew_source} | ref {reference_key}:{reference_channel}")

graph.plot_calibration_shift_summary(
    summary_tables,
    labels=plot_labels,
    reference_channel=channel_skew["reference_channel"],
)


## Inspect One Channel Fit


In [ ]:
channel = 12


def read_trace(result_group, dataset_path, column):
    return result_group[dataset_path][:, column]


with h5py.File(output_path, "r") as h5_file:
    result_group = calibration_result_group(h5_file)
    channel_indices = result_group["channels/index"][()]
    matching_columns = np.flatnonzero(channel_indices == channel)

    if len(matching_columns) == 0:
        raise ValueError(f"Channel {channel} is not present in {channel_indices.tolist()}")

    column = int(matching_columns[0])
    time_ns = h5_file["calibration/axes/time_ns"][:]

    traces = {
        "measured_trace": read_trace(result_group, "fit/measured_trace", column),
        "reference_trace": read_trace(result_group, "fit/reference_trace", column),
        "aligned_reference_trace": read_trace(
            result_group,
            "aligned/reference_trace",
            column,
        ),
        "irf_trace": read_trace(result_group, "fit/irf_trace", column),
        "aligned_irf_trace": read_trace(result_group, "aligned/irf_trace", column),
        "fitted_trace": read_trace(result_group, "fit/fitted_trace", column),
    }

    tau_ns = float(result_group["fit/tau_ns"][column])
    delay_ns = float(result_group["timing/delay_correction_ns"][column])
    residual_error = float(result_group["fit/residual_error"][column])

fig, ax = plt.subplots(figsize=(10, 4))
graph.plot_calibration_fit_traces(
    time_ns,
    traces,
    title=f"Channel {channel}: tau={tau_ns:.2f} ns, delay={delay_ns:.2f} ns, RMSE={residual_error:.3g}",
    ax=ax,
)


## HDF5 Structure


In [ ]:
print(output_path)
show_h5_structure_html(output_path)
